# Práctica 3: Filtros de Kalman en Python (Lineal, EKF y UKF)

En esta práctica implementaremos desde cero los tres algoritmos clave de la familia de Kalman. Trabajaremos bajo un enfoque modular y orientado a objetos, ideal para comprender el flujo de control y las operaciones de álgebra lineal matricial involucradas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm # Utilizado para la raíz de matrices en el UKF

# Configuración visual de las gráficas
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

---
## 1. Implementación del Filtro de Kalman Lineal (KF)

### Problema:
Hacer el seguimiento de un vehículo que se desplaza en un plano 2D con un modelo de **Velocidad Constante** sujeto a aceleraciones aleatorias (ruido de proceso). 

El vector de estado es:
$$\mathbf{x} = [p_x, p_y, v_x, v_y]^T$$

Solo podemos medir la posición ruidosa $(p_x, p_y)$ usando un sensor GPS.

In [ ]:
class KalmanFilter:
    def __init__(self, F, B, H, Q, R, x0, P0):
        # Matrices del sistema
        self.F = np.array(F, dtype=float)
        self.B = np.array(B, dtype=float) if B is not None else None
        self.H = np.array(H, dtype=float)
        self.Q = np.array(Q, dtype=float)
        self.R = np.array(R, dtype=float)
        
        # Creencia inicial
        self.x = np.array(x0, dtype=float).reshape(-1, 1)
        self.P = np.array(P0, dtype=float)
        
    def predict(self, u=None):
        # 1. Predicción del estado
        if self.B is not None and u is not None:
            u_val = np.array(u, dtype=float).reshape(-1, 1)
            self.x = self.F @ self.x + self.B @ u_val
        else:
            self.x = self.F @ self.x
            
        # 2. Predicción de la covarianza
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x, self.P
        
    def update(self, z):
        z_val = np.array(z, dtype=float).reshape(-1, 1)
        
        # Ecuaciones de corrección
        y = z_val - self.H @ self.x                  # Innovación
        S = self.H @ self.P @ self.H.T + self.R      # Covarianza de innovación
        K = self.P @ self.H.T @ np.linalg.inv(S)     # Ganancia de Kalman
        
        # Actualización de la creencia posterior
        self.x = self.x + K @ y
        I = np.eye(self.P.shape[0])
        self.P = (I - K @ self.H) @ self.P
        return self.x, self.P

### Simulación y Resultados del KF Lineal

In [ ]:
# Configuración de la simulación
dt = 0.5
pasos_tiempo = 40

# Matrices del modelo físico
F = np.array([
    [1, 0, dt, 0],
    [0, 1, 0, dt],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])

H = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
])

# Covarianza de proceso (ruido dinámico)
Q = np.eye(4) * 0.05
# Covarianza de medición (ruido GPS)
R = np.eye(2) * 4.0

# Estado inicial real y creencia inicial del filtro
x_real = np.array([0, 0, 1.5, 1.0]).reshape(-1, 1)
x_estimado_inicial = np.array([0.5, -0.5, 0.0, 0.0]) # con cierto error
P_inicial = np.eye(4) * 10.0

kf = KalmanFilter(F, None, H, Q, R, x_estimado_inicial, P_inicial)

# Listas para guardar resultados
trayectoria_real = []
mediciones = []
trayectoria_estimada = []

for t in range(pasos_tiempo):
    # 1. Simular la dinámica real con ruido de proceso
    proceso_ruido = np.random.multivariate_normal(np.zeros(4), Q).reshape(-1, 1)
    x_real = F @ x_real + proceso_ruido
    
    # 2. Generar medición ruidosa
    medida_ruido = np.random.multivariate_normal(np.zeros(2), R).reshape(-1, 1)
    z = H @ x_real + medida_ruido
    
    # 3. Filtrar
    kf.predict()
    x_est, _ = kf.update(z)
    
    # Guardar datos para graficar
    trayectoria_real.append(x_real.copy().flatten())
    mediciones.append(z.flatten())
    trayectoria_estimada.append(x_est.flatten())

trayectoria_real = np.array(trayectoria_real)
mediciones = np.array(mediciones)
trayectoria_estimada = np.array(trayectoria_estimada)

# Graficar comparación de trayectorias
plt.figure(figsize=(12, 7))
plt.plot(trayectoria_real[:, 0], trayectoria_real[:, 1], 'g-', linewidth=2.5, label='Trayectoria Real (Ground Truth)')
plt.scatter(mediciones[:, 0], mediciones[:, 1], color='red', alpha=0.5, s=25, label='Mediciones de GPS Ruidosas')
plt.plot(trayectoria_estimada[:, 0], trayectoria_estimada[:, 1], 'b--', linewidth=2, label='Filtro de Kalman')
plt.title('Seguimiento 2D con Filtro de Kalman Lineal', pad=15)
plt.xlabel('Posición X (m)')
plt.ylabel('Posición Y (m)')
plt.legend(frameon=True, facecolor='white')
plt.axis('equal')
plt.show()

---
## 2. Comparación No Lineal: EKF vs UKF

### El Desafío No Lineal (Seguimiento de Radar):
Un objeto se mueve en 1D horizontalmente a gran altura, y un sensor radar en el origen $(0,0)$ mide la **distancia radial** ($r$) mediante una ecuación altamente no lineal:
$$r = h(x) = \sqrt{x_{\text{pos}}^2 + y_{\text{alt}}^2}$$

El estado del sistema es:
$$\mathbf{x} = [x_{\text{pos}}, v_x]^T$$

La altura $y_{\text{alt}} = 1000$ m se asume constante conocida.

#### Derivación de Jacobiano para el EKF:
El jacobiano de la medición $\mathbf{H}$ es:
$$\mathbf{H} = \frac{\partial h(\mathbf{x})}{\partial \mathbf{x}} = \begin{bmatrix} \frac{x_{\text{pos}}}{\sqrt{x_{\text{pos}}^2 + y_{\text{alt}}^2}} & 0 \end{bmatrix}$$

Implementemos tanto el EKF como el UKF para ver cuál se comporta mejor ante esta medición no lineal a medida que el objeto se acerca y aleja.

In [ ]:
# Parámetros del problema del Radar
y_alt = 1000.0   # Altura fija del objetivo en metros
dt_r = 0.1
Q_radar = np.array([[0.1, 0], [0, 0.01]])  # Ruido dinámico muy bajo
R_radar = np.array([[25.0]])                # Ruido del radar: varianza de 25 metros

F_radar = np.array([
    [1.0, dt_r],
    [0.0, 1.0]
])

def h_radar(x_state):
    # Función de observación no lineal
    pos_x = x_state[0, 0]
    return np.array([[np.sqrt(pos_x**2 + y_alt**2)]])

def H_jacobian_radar(x_state):
    # Jacobiano de h(x) respecto al estado
    pos_x = x_state[0, 0]
    den = np.sqrt(pos_x**2 + y_alt**2)
    return np.array([[pos_x / den, 0.0]])

### Implementación del EKF

In [ ]:
class ExtendedKalmanFilter:
    def __init__(self, F, h_func, H_jac, Q, R, x0, P0):
        self.F = F
        self.h = h_func
        self.H_jacobian = H_jac
        self.Q = Q
        self.R = R
        self.x = np.array(x0, dtype=float).reshape(-1, 1)
        self.P = np.array(P0, dtype=float)
        
    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        
    def update(self, z):
        z_val = np.array(z, dtype=float).reshape(-1, 1)
        
        # Evaluar no linealidades y Jacobiano en la estimación actual
        h_val = self.h(self.x)
        H_mat = self.H_jacobian(self.x)
        
        y = z_val - h_val
        S = H_mat @ self.P @ H_mat.T + self.R
        K = self.P @ H_mat.T @ np.linalg.inv(S)
        
        self.x = self.x + K @ y
        I = np.eye(self.P.shape[0])
        self.P = (I - K @ H_mat) @ self.P

### Implementación del UKF (Unscented Kalman Filter)

In [ ]:
class UnscentedKalmanFilter:
    def __init__(self, F, h_func, Q, R, x0, P0, alpha=0.001, beta=2.0, kappa=0.0):
        self.F = F
        self.h = h_func
        self.Q = Q
        self.R = R
        self.x = np.array(x0, dtype=float).reshape(-1, 1)
        self.P = np.array(P0, dtype=float)
        
        self.n = self.x.shape[0]
        
        # Parámetros de la Transformación Unscented
        self.alpha = alpha
        self.beta = beta
        self.kappa = kappa
        self.lambd = (alpha**2) * (self.n + kappa) - self.n
        
        # Calcular pesos de puntos Sigma
        self.weights_m = np.zeros(2 * self.n + 1)
        self.weights_c = np.zeros(2 * self.n + 1)
        
        self.weights_m[0] = self.lambd / (self.n + self.lambd)
        self.weights_c[0] = self.weights_m[0] + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.weights_m[i] = 1.0 / (2 * (self.n + self.lambd))
            self.weights_c[i] = self.weights_m[i]
            
    def generar_puntos_sigma(self):
        # Raíz cuadrada de la matriz P multiplicada por escala
        val = (self.n + self.lambd) * self.P
        # Asegurar simetría para evitar problemas numéricos menores
        val = 0.5 * (val + val.T)
        raiz_P = sqrtm(val)
        
        sigmas = np.zeros((self.n, 2 * self.n + 1))
        sigmas[:, 0] = self.x.flatten()
        
        for i in range(self.n):
            sigmas[:, i + 1] = self.x.flatten() + raiz_P[:, i]
            sigmas[:, i + 1 + self.n] = self.x.flatten() - raiz_P[:, i]
            
        return sigmas
        
    def predict(self):
        # 1. Generar puntos sigma en t-1
        sigmas = self.generar_puntos_sigma()
        
        # 2. Propagar puntos sigma a través de la función de transición (F lineal en este caso)
        sigmas_pred = self.F @ sigmas
        
        # 3. Reconstruir media predicha a partir de los puntos sigma
        self.x = np.zeros((self.n, 1))
        for i in range(2 * self.n + 1):
            self.x += self.weights_m[i] * sigmas_pred[:, i].reshape(-1, 1)
            
        # 4. Reconstruir covarianza predicha
        self.P = np.zeros((self.n, self.n))
        for i in range(2 * self.n + 1):
            diff = (sigmas_pred[:, i].reshape(-1, 1) - self.x)
            self.P += self.weights_c[i] * (diff @ diff.T)
        self.P += self.Q
        
    def update(self, z):
        z_val = np.array(z, dtype=float).reshape(-1, 1)
        m = z_val.shape[0]
        
        # 1. Volver a generar puntos sigma a partir de la predicción actual
        sigmas_pred = self.generar_puntos_sigma()
        
        # 2. Propagar puntos sigma por la función de observación no lineal h(x)
        sigmas_z = np.zeros((m, 2 * self.n + 1))
        for i in range(2 * self.n + 1):
            sigmas_z[:, i] = self.h(sigmas_pred[:, i].reshape(-1, 1)).flatten()
            
        # 3. Media predicha de la medición
        z_pred = np.zeros((m, 1))
        for i in range(2 * self.n + 1):
            z_pred += self.weights_m[i] * sigmas_z[:, i].reshape(-1, 1)
            
        # 4. Calcular covarianzas de innovación y cruzada
        S = np.zeros((m, m))
        Pxz = np.zeros((self.n, m))
        
        for i in range(2 * self.n + 1):
            diff_z = sigmas_z[:, i].reshape(-1, 1) - z_pred
            diff_x = sigmas_pred[:, i].reshape(-1, 1) - self.x
            
            S += self.weights_c[i] * (diff_z @ diff_z.T)
            Pxz += self.weights_c[i] * (diff_x @ diff_z.T)
            
        S += self.R
        
        # 5. Ganancia de Kalman y corrección posterior
        K = Pxz @ np.linalg.inv(S)
        
        self.x = self.x + K @ (z_val - z_pred)
        self.P = self.P - K @ S @ K.T

### Simulación Comparativa: EKF vs UKF

In [ ]:
# Inicializaciones físicas del objeto
x_real_radar = np.array([[0.0], [90.0]]) # Empieza en x=0m, velocidad 90 m/s (324 km/h)

# Creencias iniciales erróneas a propósito
x0_filtro = np.array([[100.0], [75.0]])
P0_filtro = np.eye(2) * 500.0

ekf = ExtendedKalmanFilter(F_radar, h_radar, H_jacobian_radar, Q_radar, R_radar, x0_filtro, P0_filtro)
ukf = UnscentedKalmanFilter(F_radar, h_radar, Q_radar, R_radar, x0_filtro, P0_filtro, alpha=0.1, beta=2.0, kappa=0.0)

hist_real = []
hist_mediciones = []
hist_ekf = []
hist_ukf = []

pasos_sim = 80

for step in range(pasos_sim):
    # 1. Física real del objeto
    x_real_radar = F_radar @ x_real_radar + np.random.multivariate_normal(np.zeros(2), Q_radar).reshape(-1, 1)
    
    # 2. Medida no lineal ruidosa del sensor
    z_radar = h_radar(x_real_radar) + np.random.normal(0, np.sqrt(R_radar[0, 0]))
    
    # 3. Correr filtros predict & update
    ekf.predict()
    ekf.update(z_radar)
    
    ukf.predict()
    ukf.update(z_radar)
    
    # Registrar historial
    hist_real.append(x_real_radar.flatten())
    hist_mediciones.append(z_radar.flatten())
    hist_ekf.append(ekf.x.flatten())
    hist_ukf.append(ukf.x.flatten())

hist_real = np.array(hist_real)
hist_mediciones = np.array(hist_mediciones)
hist_ekf = np.array(hist_ekf)
hist_ukf = np.array(hist_ukf)

# Graficar comparación de estimación de posición
plt.figure(figsize=(12, 6.5))
plt.plot(hist_real[:, 0], label='Posición Real (Ground Truth)', color='green', linewidth=2.5)
plt.plot(hist_ekf[:, 0], label='Extended Kalman Filter (EKF)', color='red', linestyle='--', linewidth=1.8)
plt.plot(hist_ukf[:, 0], label='Unscented Kalman Filter (UKF)', color='blue', linestyle='-.', linewidth=2.0)
plt.title('Comparación de Estimación de Posición Horizontal: EKF vs UKF en Radar No Lineal', pad=15)
plt.xlabel('Paso de Tiempo')
plt.ylabel('Posición X (m)')
plt.legend(frameon=True, facecolor='white')
plt.show()

# Calcular y graficar el error cuadrático medio acumulado
error_ekf = np.abs(hist_ekf[:, 0] - hist_real[:, 0])
error_ukf = np.abs(hist_ukf[:, 0] - hist_real[:, 0])

plt.figure(figsize=(12, 5))
plt.plot(error_ekf, label='Error Absoluto EKF', color='red', alpha=0.75)
plt.plot(error_ukf, label='Error Absoluto UKF', color='blue', alpha=0.75)
plt.title('Error de Estimación Absoluto EKF vs UKF', pad=15)
plt.xlabel('Paso de Tiempo')
plt.ylabel('Error Absoluto (m)')
plt.legend(frameon=True, facecolor='white')
plt.show()

print(f"🎯 Error Medio Absoluto EKF: {np.mean(error_ekf):.3f} metros")
print(f"🎯 Error Medio Absoluto UKF: {np.mean(error_ukf):.3f} metros")
print(f"El UKF demuestra mayor convergencia y menor error gracias a la Transformación Unscented de segundo orden.")